# DPIRD weather data ingestion
Port of Mapping/DPIRD/preprocess/preprocess.py 

In [0]:
from pathlib import Path
import os, yaml, gzip, tarfile, shutil, pytz
import pandas as pd
import numpy as np
import xarray as xr
from concurrent.futures import ThreadPoolExecutor

## Define paths

In [0]:
def load_yaml(path):
    with open(path, "r") as f:
        return yaml.safe_load(f)
    
config= load_yaml('configs/config.yaml')
DPIRD_path_config= config['DPIRD_data_path']
raw_tar_path= Path(DPIRD_path_config['raw-file'])
untar_folder= Path(DPIRD_path_config['untar-folder'])

## Untar actions

In [0]:
"""
Unzip DPIRD station data by organizing into YYYY/MM/station.csv files. Untar then remove remnant zipped files
"""
if not raw_tar_path.exists():
    raise Exception(f"Check config.yaml: {raw_tar_path} does not exist.")

if not untar_folder.exists():
    print(f"Creating {untar_folder} and extracting tar... this may take a moment.")
    untar_folder.mkdir(parents=True, exist_ok=True)
    with tarfile.open(raw_tar_path, "r:gz") as tar:
        tar.extractall(path=untar_folder)

else:
    print(f"Directory {untar_folder} already exists. Skipping extraction...")

def process_station_file_in_place(file_path):
    """ 
    Unzip (if needed), map YYYYMM to YYYY/MM/, and save as .csv
    Expected after first untar, folder is "202112", we want to map this to "2021/12
    """  
    file_path= Path(file_path)
    yyyymm = file_path.parent.name
    if len(yyyymm) != 6 or not yyyymm.isdigit():
        return  

    year, month = yyyymm[:4], yyyymm[4:]
    dest_dir = untar_folder / year / month
    dest_dir.mkdir(parents=True, exist_ok=True)
    
    station_name = file_path.name.replace('.csv.gz', '.csv')
    dest_file_path = dest_dir / station_name

    # Unzip if gz and output to the new YYYY/MM struct. Delete .gz in-place after second untar
    if file_path.name.endswith('.gz'):
        with gzip.open(file_path, 'rb') as f_in, open(dest_file_path, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
        os.remove(file_path) 
    
    # Or just Move if already a csv, if the source and dest are actually different
    elif file_path.name.endswith('.csv'):
        if file_path != dest_file_path: 
            shutil.move(file_path, dest_file_path)

"""Multi-threaded execution to get all files in YYYYMM subfolders"""
all_dirs = [d for d in untar_folder.rglob('*') if d.is_dir() and len(d.name) == 6 and d.name.isdigit()]
target_files = []
for folder in all_dirs:
    target_files.extend(list(folder.glob('*.*')))

if target_files:
    print(f"Processing {len(target_files)} files using ThreadPoolExecutor...")
    with ThreadPoolExecutor(max_workers=os.cpu_count()) as executor:
        executor.map(process_station_file_in_place, target_files)
    
    # Clean up old empty YYYYMM folders. If destination parent is empty remove it
    for folder in all_dirs:
        if folder.exists() and not any(folder.iterdir()):
            folder.rmdir()
            try:
                if not any(folder.parent.iterdir()) and folder.parent != untar_folder:
                    folder.parent.rmdir()
            except Exception:
                pass

print(f"Extraction and restructuring complete at: {untar_folder}")
